# WSDM KKBox Churn Prediction

## Machine Learning Model Development & Evaluation

This notebook develops, tunes, and evaluates multiple machine learning models using the engineered feature dataset created in the feature engineering pipeline.

### Objectives

- Establish a Logistic Regression baseline
- Optimize CatBoost, LightGBM and XGBoost using Optuna
- Train CatBoost, LightGBM, and XGBoost
- Compare model performance using ROC AUC
- Select the best model for deployment

### Models

- Logistic Regression
- CatBoost
- LightGBM
- XGBoost

### Evaluation Metric

- ROC AUC

## Workflow

1. Load engineered dataset

2. Train Logistic Regression baseline

3. Optimize CatBoost

4. Optimize LightGBM

5. Optimize XGBoost

6. Compare model performance

7. Select best model

8. Save final model

## Import Libraries

In [ ]:
# Install the required packages if they are not already available in your environment

import catboost
import optuna
import pandas as pd
import polars as pl
from catboost import CatBoostClassifier, Pool
from pathlib import Path
import polars.selectors as cs
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

SEED = 42

##  Environment Configuration

In [ ]:




DATA_PATH = Path("data")

MODEL_PATH = Path("models")

MODEL_PATH.mkdir(exist_ok=True)

## Load Engineered Dataset

In [ ]:
final_train_clean = pl.read_parquet(DATA_PATH / "final_train_clean.parquet")
# if its csv
# final_train_clean = pl.read_csv(DATA_PATH / "final_train_clean.csv")
print(final_train_clean.shape)

In [ ]:
# Categorical Features
col_categorical = final_train_clean.select(cs.by_dtype(pl.Categorical))
col_categorical

city,gender,registered_via,last_payment_method
cat,cat,cat,cat
"""5""","""male""","""3""","""Unknown"""
"""13""","""male""","""3""","""36"""
"""13""","""male""","""3""","""15"""
"""1""","""Unknown""","""7""","""41"""
"""13""","""female""","""7""","""41"""
…,…,…,…
"""13""","""male""","""7""","""40"""
"""1""","""Unknown""","""7""","""41"""
"""1""","""Unknown""","""7""","""41"""


## Logistic Regression Baseline

In [ ]:
# =============================
# Load Data
# =============================

df = final_train_clean.to_pandas()

# =============================
# Target & Features
# =============================

target_column = "is_churn"
columns_to_drop_from_features = [target_column, "msno"]

X = df.drop(columns=columns_to_drop_from_features)
y = df[target_column].astype("int8")

# =============================
# Categorical Columns
# =============================

categorical_features = [
    "city",
    "registered_via",
    "last_payment_method",
    "gender",
]

# Convert categorical columns to string
for col in categorical_features:
    X[col] = X[col].astype(str)

# =============================
# Numerical Columns
# =============================

numerical_features = [
    col for col in X.columns
    if col not in categorical_features
]

# Optional: reduce memory usage
for col in numerical_features:
    X[col] = X[col].astype("float32")

# =============================
# Train / Validation / Test Split
# =============================

# 80% Train+Validation / 20% Test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=seed,
)

# 80% of training / 20% validation
# Final:
# Train = 64%
# Validation = 16%
# Test = 20%

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    stratify=y_train,
    random_state=seed,
)

# =============================
# Preprocessing
# =============================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features,
        ),
        (
            "cat",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=True,
            ),
            categorical_features,
        ),
    ]
)

# =============================
# Logistic Regression
# =============================

model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor),
        (
            "classifier",
            LogisticRegression(
                solver="saga",
                max_iter=900,
                tol=1e-3,
                random_state=seed,
                class_weight="balanced",
                n_jobs=-1,
            ),
        ),
    ]
)

# =============================
# Train Model
# =============================

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['bd', 'age_missing',
                                                   'registration_year',
                                                   'registration_month',
                                                   'member_tenure_days',
                                                   'transaction_count',
                                                   'last_paid', 'avg_paid',
                                                   'total_paid', 'std_paid',
                                                   'last_list_price',
                                                   'avg_list_price',
                                                   'std_list_price',
                                                   'num_price_changes',
                                                   'last_plan_days',
                                                   'avg_plan_days',
                                                   'std_pla...
                                                   'auto_renew_rate',
                                                   'num_auto_renew',
                                                   'last_auto_renew',
                                                   'cancel_rate', 'num_cancel',
                                                   'last_cancel',
                                                   'num_payment_method', ...]),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['city', 'registered_via',
                                                   'last_payment_method',
                                                   'gender'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=900,
                                    n_jobs=-1, random_state=42, solver='saga',
                                    tol=0.001))])

### Logistic Regression Validation / Test Performance


In [ ]:
# ============================================================
# Validation Predictions
# ============================================================

y_val_pred = model.predict(X_val)
y_val_prob = model.predict_proba(X_val)[:, 1]

print("=" * 60)
print("Validation Performance")
print("=" * 60)

print(f"Accuracy : {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred):.4f}")
print(f"Recall   : {recall_score(y_val, y_val_pred):.4f}")
print(f"F1 Score : {f1_score(y_val, y_val_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_val, y_val_prob):.4f}")

print("\nClassification Report")
print(classification_report(y_val, y_val_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_val, y_val_pred))

Validation Performance
Accuracy : 0.9602
Precision: 0.7163
Recall   : 0.9224
F1 Score : 0.8064
ROC AUC  : 0.9836

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.96      0.98    141381
           1       0.72      0.92      0.81     13973

    accuracy                           0.96    155354
   macro avg       0.85      0.94      0.89    155354
weighted avg       0.97      0.96      0.96    155354


Confusion Matrix
[[136275   5106]
 [  1084  12889]]


In [ ]:
# ============================================================
# Test Predictions (Unseen Data)
# ============================================================

y_test_pred = model.predict(X_test)
y_test_prob = model.predict_proba(X_test)[:, 1]

print("=" * 60)
print("Final Test Performance")
print("=" * 60)

print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_test_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, y_test_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_test_prob):.4f}")

print("\nClassification Report")
print(classification_report(y_test, y_test_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_test_pred))

Final Test Performance
Accuracy : 0.9591
Precision: 0.7092
Recall   : 0.9252
F1 Score : 0.8029
ROC AUC  : 0.9836

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.96      0.98    176726
           1       0.71      0.93      0.80     17466

    accuracy                           0.96    194192
   macro avg       0.85      0.94      0.89    194192
weighted avg       0.97      0.96      0.96    194192


Confusion Matrix
[[170099   6627]
 [  1307  16159]]


In [ ]:
val_prob = model.predict_proba(X_val)[:, 1]
test_prob = model.predict_proba(X_test)[:, 1]

print("Validation Log Loss:", log_loss(y_val, val_prob))
print("Test Log Loss:", log_loss(y_test, test_prob))

## CatBoost Hyperparameter Optimization


In [ ]:
# ==========================================================
# STEP 1 - Create a small Optuna dataset (7% stratified)
# ==========================================================

optuna_df = (
    final_train_clean
    .group_by("is_churn")
    .map_groups(lambda df: df.sample(fraction=0.07, seed=SEED))
)

print("Optuna dataset:", optuna_df.shape)


# ==========================================================
# STEP 2 - Convert to pandas
# ==========================================================

optuna_df = optuna_df.to_pandas()


# ==========================================================
# STEP 3 - Separate Target, Features and Customer ID
# ==========================================================

TARGET = "is_churn"

# Save customer IDs
msno = optuna_df["msno"]

# Drop target and identifier from model features
X = optuna_df.drop(columns=[TARGET, "msno"])

y = optuna_df[TARGET]


# ==========================================================
# STEP 4 - Train / Validation / Test Split
# ==========================================================

# First split: 80% train+validation, 20% test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, msno_train, msno_test = train_test_split(
    X,
    y,
    msno,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

# Second split: 80% of the remaining data for training,
# 20% for validation

X_train, X_valid, y_train, y_valid, msno_train, msno_valid = train_test_split(
    X_train,
    y_train,
    msno_train,
    test_size=0.20,
    stratify=y_train,
    random_state=SEED
)


# ==========================================================
# STEP 5 - Detect categorical columns
# ==========================================================

categorical_features = (
    X_train.select_dtypes(
        include=["object", "category"]
    )
    .columns
    .tolist()
)

print(categorical_features)


# ==========================================================
# STEP 6 - Create CatBoost Pools
# ==========================================================

from catboost import Pool

train_pool = Pool(
    X_train,
    y_train,
    cat_features=categorical_features
)

valid_pool = Pool(
    X_valid,
    y_valid,
    cat_features=categorical_features
)

test_pool = Pool(
    X_test,
    y_test,
    cat_features=categorical_features
)


# ==========================================================
# STEP 7 - Optuna Objective
# ==========================================================

import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

def objective(trial):

    params = {

        "iterations": 5000,

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),

        "depth": trial.suggest_int("depth", 4, 8),

        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1,
            20
        ),

        "random_strength": trial.suggest_float(
            "random_strength",
            0,
            5
        ),

        "bagging_temperature": trial.suggest_float(
            "bagging_temperature",
            0,
            10
        ),

        "border_count": trial.suggest_int(
            "border_count",
            64,
            255
        ),

        "loss_function": "Logloss",

        "eval_metric": "AUC",

        "task_type": "GPU",

        "devices": "0",

        "verbose": False,

        "random_seed": SEED
    }

    model = CatBoostClassifier(**params)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        early_stopping_rounds=150,
        use_best_model=True
    )

    pred = model.predict_proba(valid_pool)[:,1]

    auc = roc_auc_score(y_valid, pred)

    return auc


# ==========================================================
# STEP 8 - Run Optuna
# ==========================================================

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)

study.optimize(
    objective,
    n_trials=30,
    show_progress_bar=True
)


print(study.best_value)
print(study.best_params)

## CatBoost Model

In [ ]:
# ==========================================================
# Convert to pandas
# ==========================================================

df = final_train_clean.to_pandas()

# ==========================================================
# Keep msno separately
# ==========================================================

TARGET = "is_churn"

msno = df["msno"]

X = df.drop(columns=[TARGET, "msno"])
y = df[TARGET]

# ==========================================================
# Train / Validation / Test Split
# 64% / 16% / 20%
# ==========================================================

X_train, X_test, y_train, y_test, msno_train, msno_test = train_test_split(
    X,
    y,
    msno,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)

X_train, X_valid, y_train, y_valid, msno_train, msno_valid = train_test_split(
    X_train,
    y_train,
    msno_train,
    test_size=0.20,
    stratify=y_train,
    random_state=SEED,
)

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

# ==========================================================
# Categorical Features
# ==========================================================

categorical_features = [
    "city",
    "gender",
    "registered_via",
    "last_payment_method",
]

# Convert categorical columns to string
for col in categorical_features:
    X_train[col] = X_train[col].astype(str)
    X_valid[col] = X_valid[col].astype(str)
    X_test[col] = X_test[col].astype(str)

# ==========================================================
# Pools
# ==========================================================

train_pool = Pool(
    X_train,
    y_train,
    cat_features=categorical_features
)

valid_pool = Pool(
    X_valid,
    y_valid,
    cat_features=categorical_features
)

test_pool = Pool(
    X_test,
    y_test,
    cat_features=categorical_features
)

# ==========================================================
# CatBoost Model
# ==========================================================

model = CatBoostClassifier(

    iterations=5000,

    learning_rate=0.012184186502221764,

    depth=8,

    l2_leaf_reg=12.421185223120967,

    random_strength=3.540362888980227,

    bagging_temperature=0.20584494295802447,

    border_count=250,

    loss_function="Logloss",

    eval_metric="AUC",

    random_seed=SEED,

    task_type="GPU",

    devices="0",

    verbose=200,
)

# ==========================================================
# Train
# ==========================================================

model.fit(
    train_pool,
    eval_set=valid_pool,
    early_stopping_rounds=200,
    use_best_model=True,
)

### CatBoost Validation / Test Performance

In [ ]:
# ==========================================================
# Validation
# ==========================================================

val_prob = model.predict_proba(valid_pool)[:, 1]
val_pred = model.predict(valid_pool)

print("=" * 60)
print("Validation Performance")
print("=" * 60)

print(f"AUC      : {roc_auc_score(y_valid, val_prob):.6f}")
print(f"LogLoss  : {log_loss(y_valid, val_prob):.6f}")
print(f"Accuracy : {accuracy_score(y_valid, val_pred):.6f}")
print(f"Precision: {precision_score(y_valid, val_pred):.6f}")
print(f"Recall   : {recall_score(y_valid, val_pred):.6f}")
print(f"F1 Score : {f1_score(y_valid, val_pred):.6f}")

print("\nClassification Report")
print(classification_report(y_valid, val_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_valid, val_pred))

# ==========================================================
# Test
# ==========================================================

test_prob = model.predict_proba(test_pool)[:, 1]
test_pred = model.predict(test_pool)

print("=" * 60)
print("Final Test Performance")
print("=" * 60)

print(f"AUC      : {roc_auc_score(y_test, test_prob):.6f}")
print(f"LogLoss  : {log_loss(y_test, test_prob):.6f}")
print(f"Accuracy : {accuracy_score(y_test, test_pred):.6f}")
print(f"Precision: {precision_score(y_test, test_pred):.6f}")
print(f"Recall   : {recall_score(y_test, test_pred):.6f}")
print(f"F1 Score : {f1_score(y_test, test_pred):.6f}")

print("\nClassification Report")
print(classification_report(y_test, test_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, test_pred))

# ==========================================================
# Save Model
# ==========================================================

model.save_model("catboost_final.cbm")

print("Model saved successfully.")

Validation Performance
AUC      : 0.996869
LogLoss  : 0.040077
Accuracy : 0.983270
Precision: 0.898305
Recall   : 0.917913
F1 Score : 0.908003

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    141381
           1       0.90      0.92      0.91     13973

    accuracy                           0.98    155354
   macro avg       0.95      0.95      0.95    155354
weighted avg       0.98      0.98      0.98    155354


Confusion Matrix
[[139929   1452]
 [  1147  12826]]
Final Test Performance
AUC      : 0.996767
LogLoss  : 0.040423
Accuracy : 0.982806
Precision: 0.893269
Recall   : 0.918585
F1 Score : 0.905750

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    176726
           1       0.89      0.92      0.91     17466

    accuracy                           0.98    194192
   macro avg       0.94      0.95      0.95    194192
weighted avg    

## LightGBM Hyperparameter Optimization

In [ ]:


# ========================
# Create optuna dataset
# ========================

optuna_df = (
    final_train_clean
    .group_by('is_churn')
    .map_groups(lambda df: df.sample(fraction= 0.07, seed= SEED))
)

print(optuna_df.shape)

# =============================
# Convert to pandas
# =============================
# lightgbm works better with pandas

optuna_df = optuna_df.to_pandas()

# =============================
# Target / Features
# =============================

target = 'is_churn'
msno_ids = optuna_df['msno']
X = optuna_df.drop(columns= [target, 'msno'])
y = optuna_df[target]

# ======================================
# Train 64% / validation 16% / test 20%
# ======================================

X_train, X_test, y_train, y_test, msno_train, msno_test = train_test_split(
    X,
    y,
    msno_ids,
    test_size= 0.20,
    stratify= y,
    random_state= SEED,
)

X_train, X_val, y_train, y_val, msno_train, msno_valid = train_test_split(
    X_train,
    y_train,
    msno_train,
    test_size= 0.20,
    stratify= y_train,
    random_state= SEED
)

# ==============================
# Convert Categorical Columns
# ==============================

categorical_features = [
     "city",
    "gender",
    "registered_via",
    "last_payment_method"
]
for col in categorical_features:
  X_train[col] = X_train[col].astype('category')
  X_val[col] = X_val[col].astype('category')
  X_test[col] = X_test[col].astype('category')

# ==========================
# Optuna objective
# ==========================

def objective(trial):
  params = {
      'objective': 'binary',
      'metric': 'auc',
      'boosting_type': 'gbdt',
      'n_estimators': trial.suggest_int(
          'n_estimators',
          500,
          5000
      ),
      'learning_rate': trial.suggest_float(
          'learning_rate',
          0.01,
          0.30,
          log= True
      ),
      'num_leaves': trial.suggest_int(
          'num_leaves',
          31,
          255
      ),
      'max_depth': trial.suggest_int(
          'max_depth',
          4,
          8
      ),
      'min_child_samples': trial.suggest_int(
          'min_child_samples',
          20,
          200
      ),
      'feature_fraction' : trial.suggest_float(
          'feature_fraction',
          0.6,
          1.0
      ),
      'bagging_fraction' : trial.suggest_float(
          'bagging_fraction',
          0.6,
          1.0
      ),
      'bagging_freq' : trial.suggest_int(
          'bagging_freq',
          1,
          10
      ),
      'lambda_l1' : trial.suggest_int(
          'lambda_l1',
          0,
          20
      ),
      'lambda_l2' : trial.suggest_int(
          'lambda_l2',
          0,
          20
      ),
      'verbosity' : -1,
      'random_state' : SEED,
      'device' : 'gpu'
  }

  model = lgb.LGBMClassifier(
      **params
  )

  model.fit(
      X_train,
      y_train,
      eval_set= [(X_val, y_val)],
      eval_metric= 'auc',
      categorical_feature= categorical_features,
      callbacks= [
          lgb.early_stopping(150, verbose= False)
      ]
  )
  pred = model.predict_proba(X_val)[:,1]

  auc = roc_auc_score(y_val, pred)
  return auc

# ==========================================================
# STEP 7 - Run Optuna
# ==========================================================

study = optuna.create_study(

    direction="maximize",

    sampler=optuna.samplers.TPESampler(seed=SEED)

)

study.optimize(

    objective,

    n_trials=30,

    show_progress_bar=True

)

print(study.best_value)

print(study.best_params)

## LightGBM Model

In [ ]:
# =================================
# Convert to pandas
# =================================

df = final_train_clean.to_pandas()

# ================================
# keep msno seprately
# ================================

target = 'is_churn'
msno = df['msno']

X = df.drop(columns=[target, 'msno'])
y = df[target]

# ==================================
# Train 64% validation 16% test 20%
# ==================================
X_train, X_test, y_train, y_test, msno_train, msno_test = train_test_split(
    X,
    y,
    msno,
    test_size= 0.2,
    stratify=y,
    random_state= 42
)

X_train, X_valid, y_train, y_valid, msno_train, msno_valid = train_test_split(
    X_train,
    y_train,
    msno_train,
    test_size= 0.2,
    stratify=y_train,
    random_state= 42
)

print('Train', X_train.shape )
print('Validation', X_valid.shape)
print('Test', X_test.shape)

# ===============================
# Categorical Features
# ===============================

categorical_features = [
     "city",
    "gender",
    "registered_via",
    "last_payment_method"
]
for col in categorical_features:
  X_train[col] = X_train[col].astype('category')
  X_valid[col] = X_valid[col].astype('category')
  X_test[col] = X_test[col].astype('category')

# =======================================
# Lightgbm Model
# =======================================
model = lgb.LGBMClassifier(

    objective="binary",
    metric="auc",
    boosting_type="gbdt",
    device="gpu",
    random_state=42,
    verbosity=-1,
    n_estimators=4000,
    learning_rate=0.05363224402481346,
    num_leaves=124,
    max_depth=7,
    min_child_samples=121,
    feature_fraction=0.7425884906674536,
    bagging_fraction=0.8250812876941268,
    bagging_freq=4,
    lambda_l1=0,
    lambda_l2=0
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="auc",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(150),
        lgb.log_evaluation(period=100)
    ]
)
print(model.best_iteration_)

### LightGBM Validation / Test Performance

In [ ]:
# ====================================
# validation performance
# ====================================

# Probability predictions
val_prob = model.predict_proba(X_valid)[:, 1]

# Binary predictions
val_pred = (val_prob >= 0.5).astype(int)

print("="*60)
print("Validation Performance")
print("="*60)

print(f"Accuracy : {accuracy_score(y_valid, val_pred):.4f}")
print(f"Precision: {precision_score(y_valid, val_pred):.4f}")
print(f"Recall   : {recall_score(y_valid, val_pred):.4f}")
print(f"F1 Score : {f1_score(y_valid, val_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_valid, val_prob):.4f}")
print(f"Log Loss : {log_loss(y_valid, val_prob):.6f}")

print("\nClassification Report")
print(classification_report(y_valid, val_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_valid, val_pred))

# =====================================
# Test performance
# =====================================

# Probability predictions
test_prob = model.predict_proba(X_test)[:, 1]

# Binary predictions
test_pred = (test_prob >= 0.5).astype(int)

print("="*60)
print("Test Performance")
print("="*60)

print(f"Accuracy : {accuracy_score(y_test, test_pred):.4f}")
print(f"Precision: {precision_score(y_test, test_pred):.4f}")
print(f"Recall   : {recall_score(y_test, test_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, test_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, test_prob):.4f}")
print(f"Log Loss : {log_loss(y_test, test_prob):.6f}")

print("\nClassification Report")
print(classification_report(y_test, test_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, test_pred))

Validation Performance
Accuracy : 0.9830
Precision: 0.8979
Recall   : 0.9146
F1 Score : 0.9062
ROC AUC  : 0.9968
Log Loss : 0.040463

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    141381
           1       0.90      0.91      0.91     13973

    accuracy                           0.98    155354
   macro avg       0.94      0.95      0.95    155354
weighted avg       0.98      0.98      0.98    155354


Confusion Matrix
[[139927   1454]
 [  1193  12780]]
Test Performance
Accuracy : 0.9828
Precision: 0.8940
Recall   : 0.9181
F1 Score : 0.9059
ROC AUC  : 0.9968
Log Loss : 0.040643

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    176726
           1       0.89      0.92      0.91     17466

    accuracy                           0.98    194192
   macro avg       0.94      0.95      0.95    194192
weighted avg       0.98      0.98      0.9

## XGBoost Hyperparameter Optimization

In [ ]:
# ==================================
# Create optuna dataset 7%
# ==================================

optuna_df = (
    final_train_clean
    .group_by('is_churn')
    .map_groups(lambda df: df.sample(fraction= 0.07, seed = SEED))
)
print(optuna_df.shape)

# =================================
# Convert to pandas
# =================================

df = optuna_df.to_pandas()

# ==================================
# Keep msno seprately
# ==================================

TARGET = 'is_churn'
msno = df['msno']
X = df.drop(columns= [TARGET, 'msno'])
y = df[TARGET]

# =======================================
# Train 64% / Validation 16% / Test 20%
# =======================================

X_train, X_test, y_train, y_test, msno_train, msno_test = train_test_split(
    X,
    y,
    msno, # Pass msno_ids for co-splitting
    test_size= 0.20,
    stratify= y,
    random_state= SEED
)

X_train, X_valid, y_train, y_valid, msno_train, msno_valid = train_test_split(
    X_train,
    y_train,
    msno_train, # Pass msno_train for co-splitting
    test_size= 0.20,
    stratify= y_train, # Changed stratify to y_train
    random_state= SEED
)
print(X_train.shape)
print(X_valid.shape)
print(X_test.shape)

# ===============================
# Categorical Features
# ===============================
categorical_features = [
     "city",
    "gender",
    "registered_via",
    "last_payment_method"
]
for  col in categorical_features:
  X_train[col] = X_train[col].astype('category')
  X_valid[col] = X_valid[col].astype('category')
  X_test[col]  = X_test[col].astype('category')

# ===================================
# Optuna objective
# ===================================
def objective(trial):
  params = {
      'objective': 'binary:logistic',
      'eval_metric': 'auc',
      'tree_method': 'hist',
      'device': 'cuda',
      'enable_categorical': True,
      'n_estimators': 5000,
      'learning_rate': trial.suggest_float(
          'learning_rate',
          0.01,
          0.30
      ),
      'max_depth': trial.suggest_int(
          'max_depth',
          3,
          10
      ),
      'min_child_weight': trial.suggest_int(
          'min_child_weight',
          1,
          20
      ),
      'gamma': trial.suggest_int(
          'gamma',
          0,
          10
      ),
      'subsample': trial.suggest_float(
          'subsample',
          0.6,
          1.0
      ),
      'colsample_bytree': trial.suggest_float(
          'colsample_bytree',
          0.6,
          1.0
      ),
      'reg_alpha': trial.suggest_float(
          'reg_alpha',
          0,
          20
      ),
      'reg_lambda': trial.suggest_float(
          'reg_lambda',
          0,
          20
      ),
      'random_state': SEED

  }

  model = xgb.XGBClassifier(
    **params,
    early_stopping_rounds=150
)

  model.fit(
      X_train,
      y_train,
      eval_set=[(X_valid, y_valid)],
      verbose= False
  )

  pred = model.predict_proba(X_valid)[:, 1]
  auc = roc_auc_score(y_valid, pred)
  return auc

# ================================
# Run optuna
# ================================

study = optuna.create_study(
    direction= 'maximize',
    sampler= optuna.samplers.TPESampler(seed= SEED)
)

study.optimize(
    objective,
    n_trials= 30,
    show_progress_bar= True
)

print(study.best_value)
print(study.best_params)

## XGBoost Model

In [ ]:
# =======================================
# XGBoost Model
# =======================================

model = xgb.XGBClassifier(

    # Best hyperparameters
    n_estimators=5000,
    learning_rate=0.2536116052763765,
    max_depth=8,
    min_child_weight=3,
    gamma=1,
    subsample=0.9509122370961715,
    colsample_bytree=0.8544309019659351,
    reg_alpha=2.5767301684667467,
    reg_lambda=19.88216520662168,

    # Fixed parameters
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    device="cuda",
    enable_categorical=True,
    random_state=42,
    early_stopping_rounds=150
)

# =======================================
# Train Model
# =======================================

model.fit(

    X_train,
    y_train,

    eval_set=[(X_valid, y_valid)],

    verbose=100

)

print("Training Complete!")

### XGBoost Validation / Test Performance

In [ ]:
val_prob = model.predict_proba(X_valid)[:,1]
val_pred = (val_prob >= 0.5).astype(int)

print("="*60)
print("Validation Performance")
print("="*60)

print(f"Accuracy : {accuracy_score(y_valid, val_pred):.4f}")
print(f"Precision: {precision_score(y_valid, val_pred):.4f}")
print(f"Recall   : {recall_score(y_valid, val_pred):.4f}")
print(f"F1 Score : {f1_score(y_valid, val_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_valid, val_prob):.4f}")
print(f"Log Loss : {log_loss(y_valid, val_prob):.6f}")

print("\nClassification Report")
print(classification_report(y_valid, val_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_valid, val_pred))

test_prob = model.predict_proba(X_test)[:,1]
test_pred = (test_prob >= 0.5).astype(int)

print("="*60)
print("Test Performance")
print("="*60)

print(f"Accuracy : {accuracy_score(y_test, test_pred):.4f}")
print(f"Precision: {precision_score(y_test, test_pred):.4f}")
print(f"Recall   : {recall_score(y_test, test_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, test_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, test_prob):.4f}")
print(f"Log Loss : {log_loss(y_test, test_prob):.6f}")

print("\nClassification Report")
print(classification_report(y_test, test_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, test_pred))

Validation Performance
Accuracy : 0.9812
Precision: 0.8949
Recall   : 0.8967
F1 Score : 0.8958
ROC AUC  : 0.9953
Log Loss : 0.048621

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      9897
           1       0.89      0.90      0.90       978

    accuracy                           0.98     10875
   macro avg       0.94      0.94      0.94     10875
weighted avg       0.98      0.98      0.98     10875


Confusion Matrix
[[9794  103]
 [ 101  877]]
Test Performance
Accuracy : 0.9821
Precision: 0.8925
Recall   : 0.9101
F1 Score : 0.9012
ROC AUC  : 0.9960
Log Loss : 0.045644

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     12371
           1       0.89      0.91      0.90      1223

    accuracy                           0.98     13594
   macro avg       0.94      0.95      0.95     13594
weighted avg       0.98      0.98      0.98     13

## Model Comparison

The table below summarizes the performance of all machine learning models evaluated in this project on both the validation and test datasets.

The primary evaluation metric is **ROC AUC**, while Accuracy, Precision, Recall, F1 Score, and Log Loss provide additional insight into classification performance.

In [ ]:
import pandas as pd

# ==========================================================
# Model Comparison
# ==========================================================

comparison = pd.DataFrame({

    "Model": [
        "Logistic Regression",
        "CatBoost",
        "LightGBM",
        "XGBoost"
    ],

    # ---------------- Validation ----------------

    "Validation ROC AUC": [
        0.9836,
        0.996869,
        0.9968,
        0.9953
    ],

    "Validation Accuracy": [
        0.9602,
        0.983270,
        0.9830,
        0.9812
    ],

    "Validation Precision": [
        0.7163,
        0.898305,
        0.8979,
        0.8949
    ],

    "Validation Recall": [
        0.9224,
        0.917913,
        0.9146,
        0.8967
    ],

    "Validation F1": [
        0.8064,
        0.908003,
        0.9062,
        0.8958
    ],

    "Validation Log Loss": [
        0.153627,
        0.040077,
        0.040463,
        0.048621
    ],

    # ---------------- Test ----------------

    "Test ROC AUC": [
        0.9836,
        0.996767,
        0.9968,
        0.9960
    ],

    "Test Accuracy": [
        0.9591,
        0.982806,
        0.9828,
        0.9821
    ],

    "Test Precision": [
        0.7092,
        0.893269,
        0.8940,
        0.8925
    ],

    "Test Recall": [
        0.9252,
        0.918585,
        0.9181,
        0.9101
    ],

    "Test F1": [
        0.8029,
        0.905750,
        0.9059,
        0.9012
    ],

    "Test Log Loss": [
        0.156384,
        0.040423,
        0.040643,
        0.045644
    ]

})

comparison

,Model,Validation ROC AUC,Validation Accuracy,Validation Precision,Validation Recall,Validation F1,Validation Log Loss,Test ROC AUC,Test Accuracy,Test Precision,Test Recall,Test F1,Test Log Loss
0,Logistic Regression,0.983600,0.96020,0.716300,0.922400,0.806400,0.153627,0.983600,0.959100,0.709200,0.925200,0.80290,0.156384
1,CatBoost,0.996869,0.98327,0.898305,0.917913,0.908003,0.040077,0.996767,0.982806,0.893269,0.918585,0.90575,0.040423
2,LightGBM,0.996800,0.98300,0.897900,0.914600,0.906200,0.040463,0.996800,0.982800,0.894000,0.918100,0.90590,0.040643
3,XGBoost,0.995300,0.98120,0.894900,0.896700,0.895800,0.048621,0.996000,0.982100,0.892500,0.910100,0.90120,0.045644


## Best Model Selection

Model performance was primarily evaluated using **ROC AUC**, with Accuracy, Precision, Recall, F1 Score, and Log Loss used as complementary metrics.

Among the evaluated models, **LightGBM** achieved the highest ROC AUC on the independent test set while maintaining excellent performance across all evaluation metrics. Therefore, LightGBM was selected as the final production model for deployment.

In [ ]:
# ==========================================================
# Best Model Summary
# ==========================================================

best_model = comparison.loc[
    comparison["Test ROC AUC"].idxmax()
]

print("=" * 60)
print("Best Performing Model")
print("=" * 60)

print(f"Model              : {best_model['Model']}")
print(f"Validation ROC AUC : {best_model['Validation ROC AUC']:.6f}")
print(f"Test ROC AUC       : {best_model['Test ROC AUC']:.6f}")
print(f"Test Accuracy      : {best_model['Test Accuracy']:.4f}")
print(f"Test Precision     : {best_model['Test Precision']:.4f}")
print(f"Test Recall        : {best_model['Test Recall']:.4f}")
print(f"Test F1 Score      : {best_model['Test F1']:.4f}")
print(f"Test Log Loss      : {best_model['Test Log Loss']:.6f}")

Best Performing Model
Model              : LightGBM
Validation ROC AUC : 0.996800
Test ROC AUC       : 0.996800
Test Accuracy      : 0.9828
Test Precision     : 0.8940
Test Recall        : 0.9181
Test F1 Score      : 0.9059
Test Log Loss      : 0.040643


## Saving Best Model

In [ ]:
# ==========================================================
# Save Best Model
# ==========================================================

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODEL_DIR / "lightgbm_best_model.txt"

final_model.booster_.save_model(str(MODEL_PATH))

print("=" * 60)
print("Model saved successfully!")
print(f"Location: {MODEL_PATH}")

## Project Summary

Four machine learning models were trained and evaluated for customer churn prediction.

Based on the evaluation results, **LightGBM** achieved the highest ROC AUC on the independent test set while maintaining excellent performance across Accuracy, Precision, Recall, F1 Score, and Log Loss.

The trained LightGBM model is saved and will be used in the deployment notebook for inference on new customer data.